<a href="https://colab.research.google.com/github/nmwaura4/Projects/blob/main/Monty_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement




The Amblyseius montdorensis production department is consistently facing significant challenges due to recurring predator insurgency, which directly compromises our ability to reliably meet market demand. This issue leads to critical supply shortages, resulting in lost sales opportunities, potential damage to our market reputation, and ultimately, a negative impact on our revenue and profitability. Our project aims to precisely identify the key environmental and biological factors that critically influence montdorensis population dynamics and predator proliferation, enabling us to implement data-driven strategies to stabilize production, ensure consistent supply, and mitigate these substantial business risks.

Libraries

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_log_error,mean_absolute_error

In [ ]:
df=pd.read_excel('/content/Monty_3.xlsx')
df.head()

In [ ]:
#cheking the size of Data
df.shape


In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
pd.get_dummies(df,drop_first=True)

In [ ]:
for col in df.columns:
  if df[col].dtype == 'object':
    le=LabelEncoder()
    # Convert column to string type to handle mixed types (e.g., NaNs and strings)
    df[col]=le.fit_transform(df[col].astype(str))

In [ ]:
df.tail(3)

In [ ]:
# Convert boolean columns to integers (0 and 1)
for col in df.columns:
  if df[col].dtype == 'bool':
    df[col] = df[col].astype(int)

print("Boolean columns converted to integers.")
display(df.head())

In [ ]:
for col in df.columns:
  if df[col].dtype == 'object': # Corrected: check dtype and added colon
    df[col] = df[col].fillna(df[col].mode()[0]) # Corrected: fill with mode of object column
  else: # Added colon
    df[col] = df[col].fillna(df[col].mean()) # Corrected: fill numerical columns with mean

In [ ]:
df.drop(columns=['Unnamed: 41','Unnamed: 42','Unnamed: 43','Unnamed: 44'], inplace=True, errors='ignore')

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
if not missing_values.empty:
  print("Columns with remaining missing values:")
  print(missing_values)
else:
  print("No remaining missing values found in any column.")

In [ ]:
df.drop(columns=['Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21'], inplace=True, errors='ignore')
print("Dropped 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21','Unnamed: 22', columns.")
display(df.head())

In [ ]:
df.isna().sum()

In [ ]:
corr=df.corr()
corr

In [ ]:
corr = df.corr()
plt.figure(figsize=(20,10))
sns.heatmap(corr,annot=True,cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

### Predicting for the entire month of June 2026

To generate predictions for a full month, we'll follow a similar process:
1.  Define the date range for June 2026.
2.  Create a DataFrame (`X_future_month`) with feature data for each day in that range. Again, for demonstration, we'll use the mean of the training `X` data.
3.  Use the `model` to predict the 'montdorensis count (/g)' for each day.
4.  Store the results in a new DataFrame.

In [ ]:
# 1. Define the future dates for June 2026
start_date_june = '2026-06-01'
end_date_june = '2026-06-30'
future_dates_june = pd.date_range(start=start_date_june, end=end_date_june, freq='D')

# 2. Prepare future feature data (X_future_month)
# For simplicity, we'll use the mean of the existing X data for each feature
# In a real scenario, you would provide actual or forecasted feature values for each day.
X_future_month = pd.DataFrame([X.mean()] * len(future_dates_june), columns=X.columns)

# 3. Make predictions for the entire month
predictions_june = model.predict(X_future_month)

# 4. Store the results in a DataFrame
future_month_df = pd.DataFrame({
    'Date': future_dates_june,
    'Predicted_montdorensis_count': predictions_june
})

print("Predictions for June 2026:")
display(future_month_df.head(12))

In [ ]:
display(future_month_df.tail())

In [ ]:
historical_average = df['montdorensis count (/g)'].mean()

print(f"Historical Average montdorensis count (/g): {historical_average:.2f}")

# Add the historical average to the future_month_df for comparison
future_month_df['Historical_Average'] = historical_average

print("\nJune 2026 Predictions vs. Historical Average:")
display(future_month_df.head())

In [ ]:
X=df.drop(columns=['montdorensis count (/g)'])
y=df['montdorensis count (/g)']

In [ ]:
model=RandomForestRegressor()
model.fit(X,y)

# Column of importance
A higher 'Importance' value for a feature indicates that it has a greater influence on the model's predictions. In other words, changes in that feature's value are more strongly associated with changes in the 'montdorensis count (/g)'.
These scores are typically normalized so that they sum up to 1 (or 100%).

In [ ]:
column_importance=pd.DataFrame({'Feature':X.columns,'Importance':model.feature_importances_})
column_importance=column_importance.sort_values(by='Importance',ascending=False)
column_importance

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
# Corrected: Create a DataFrame with y_pred and y_test as columns
Data_frame = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print(Data_frame)

### New Feature Importances (after dropping low-importance features)

In [ ]:
column_importance = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
column_importance = column_importance.sort_values(by='Importance', ascending=False)
display(column_importance)

### Model evaluation Metrics

In [ ]:
print(f"MEA {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE {mean_squared_error(y_test, y_pred):.2f}")
print(f"RMSE {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2 {r2_score(y_test, y_pred):.4f}")

MAE (Mean Absolute Error) = 2.59: This means, on average, the model predictions for 'montdorensis count (/g)' are off by approximately 2.59 units from the actual count.
MSE (Mean Squared Error) = 38.00: This is the average of the squared differences between predicted and actual values. It gives a higher penalty to larger errors.
RMSE (Root Mean Squared Error) = 6.16: This is the square root of the MSE and is expressed in the same units as your target variable ('montdorensis count (/g)'). So, on average, the prediction error is about 6.16 montdorensis.
R2 (R-squared) = 0.9950: This is a very high score, indicating that approximately 99.50% of the variance in the 'montdorensis count (/g)' can be explained by the model's input features.

In [ ]:
# Save the processed DataFrame to a CSV file
df.to_csv('montdorensis_data.csv', index=False)
print('Data saved to montdorensis_data.csv')

In [ ]:
from google.colab import files
files.download('montdorensis_data.csv')